# 05 -- Adapters, Export, and Hybrid Pipelines

This notebook covers ecosystem integration: exporting RankFlow data in various formats, importing from standard IR/RAG formats (TREC, ranx, RAGAS), and modeling hybrid search pipelines with `MergeRankFlow`.

In [ ]:
%matplotlib inline

import json
import tempfile
from pathlib import Path

import numpy as np

from rankflow import RankFlow, MergeRankFlow, PipelineStep

## Sample data

In [ ]:
rf = RankFlow(
    ranks=np.array([[0, 1, 2, 3, 4], [2, 0, 3, 1, 4], [1, 0, 2, 3, 4]]),
    step_labels=["BM25", "Semantic", "Cross-Encoder"],
    chunk_labels=["doc_a", "doc_b", "doc_c", "doc_d", "doc_e"],
    relevant_chunks=["doc_a", "doc_c"],
    relevance_grades={"doc_a": 2, "doc_c": 1},
    scores=np.array([
        [0.95, 0.80, 0.70, 0.50, 0.30],
        [0.60, 0.92, 0.40, 0.85, 0.20],
        [0.88, 0.90, 0.75, 0.45, 0.15],
    ]),
)

---
## Part 1: Export Utilities

### Export as dictionary

In [ ]:
d = rf.to_dict()
print(f"Keys: {list(d.keys())}")
print(f"Ranks shape: {len(d['ranks'])} steps x {len(d['ranks'][0])} docs")

### Export as pandas DataFrame

In [ ]:
rf.to_dataframe()

### Export as JSON

In [ ]:
with tempfile.NamedTemporaryFile(suffix=".json", delete=False, mode="w") as f:
    rf.to_json(f.name)
    print(f"Saved to {f.name}")
    # Show first 20 lines
    with open(f.name) as r:
        lines = r.readlines()
    print("".join(lines[:20]), "..." if len(lines) > 20 else "")

---
## Part 2: RankFlow JSON Schema

RankFlow has its own JSON interchange format (v1.0) that preserves scores, source labels, and relevance grades.

In [ ]:
tmp = Path(tempfile.mkdtemp())

# Save
rf.to_rankflow_json(str(tmp / "pipeline.json"), query="What is retrieval augmented generation?")

# Inspect
with open(tmp / "pipeline.json") as f:
    data = json.load(f)
print(json.dumps(data, indent=2)[:800], "...")

In [ ]:
# Load back
rf_loaded = RankFlow.from_rankflow_json(str(tmp / "pipeline.json"))
print(f"Loaded: {rf_loaded.ranks.shape[0]} steps, {rf_loaded.ranks.shape[1]} docs")
print(f"Steps: {rf_loaded.step_labels}")
print(f"Relevant: {rf_loaded.relevant_chunks}")
rf_loaded.plot()

---
## Part 3: TREC Format

The TREC run format is the de facto standard in information retrieval research. Export a single step or import multi-step pipelines from multiple run files.

In [ ]:
# Export the last step as a TREC run
trec_path = tmp / "cross_encoder.run"
rf.to_trec_run(str(trec_path), run_id="cross_enc", step_index=-1, query_id="q1")

print("TREC run file contents:")
print(trec_path.read_text())

### Importing from TREC run files

You can load multiple run files (one per retrieval step) and optionally a qrels file for relevance judgments.

In [ ]:
# Create two run files for two steps
bm25_run = tmp / "bm25.run"
bm25_run.write_text(
    "q1 Q0 doc_a 0 0.95 bm25\n"
    "q1 Q0 doc_b 1 0.80 bm25\n"
    "q1 Q0 doc_c 2 0.70 bm25\n"
    "q1 Q0 doc_d 3 0.50 bm25\n"
    "q1 Q0 doc_e 4 0.30 bm25\n"
)

reranker_run = tmp / "reranker.run"
reranker_run.write_text(
    "q1 Q0 doc_b 0 0.92 reranker\n"
    "q1 Q0 doc_a 1 0.88 reranker\n"
    "q1 Q0 doc_d 2 0.75 reranker\n"
    "q1 Q0 doc_c 3 0.60 reranker\n"
    "q1 Q0 doc_e 4 0.15 reranker\n"
)

# Qrels file
qrels_path = tmp / "qrels.txt"
qrels_path.write_text(
    "q1 0 doc_a 2\n"
    "q1 0 doc_c 1\n"
)

rf_trec = RankFlow.from_trec_run(
    [str(bm25_run), str(reranker_run)],
    qrels_path=str(qrels_path),
    query_id="q1",
)
print(f"Loaded {rf_trec.ranks.shape[0]} steps, {rf_trec.ranks.shape[1]} docs")
print(f"Relevant: {rf_trec.relevant_chunks}")
rf_trec.plot()

---
## Part 4: ranx and RAGAS Adapters

These adapters require the optional dependencies (`pip install rankflow[ranx]` or `pip install rankflow[ragas]`).

### ranx

```python
from ranx import Run, Qrels

run1 = Run({"q1": {"doc_a": 0.9, "doc_b": 0.7}}, name="bm25")
run2 = Run({"q1": {"doc_b": 0.95, "doc_a": 0.5}}, name="reranker")
qrels = Qrels({"q1": {"doc_a": 2}})

rf = RankFlow.from_ranx([run1, run2], qrels=qrels, query_id="q1")
rf.plot()
```

### RAGAS

```python
# Works with RAGAS EvaluationDataset or list of dicts
samples = [
    {
        "user_input": "What is RAG?",
        "retrieved_contexts": ["RAG combines...", "LLMs generate..."],
        "reference_contexts": ["RAG combines..."],
    },
]
batch = RankFlow.from_ragas(samples)
```

---
## Part 5: Hybrid Pipelines with MergeRankFlow

`MergeRankFlow` models DAG-structured pipelines where multiple retrieval branches merge. This is common in hybrid search (BM25 + vector search merged via Reciprocal Rank Fusion).

In [ ]:
rng = np.random.default_rng(42)

# Two independent retrieval branches
bm25_step = PipelineStep(
    name="BM25",
    ranks=np.arange(15),
    chunk_labels=[f"doc_{i}" for i in range(15)],
)

vector_step = PipelineStep(
    name="Vector",
    ranks=np.arange(15),
    chunk_labels=[f"doc_{i}" for i in [3, 7, 11, 0, 5, 14, 2, 8, 12, 1, 6, 9, 4, 10, 13]],
)

# RRF merge step combining both branches
merged_docs = list(set(bm25_step.chunk_labels) | set(vector_step.chunk_labels))
merged_docs.sort()
rrf_step = PipelineStep(
    name="RRF Merge",
    ranks=rng.permutation(len(merged_docs)),
    chunk_labels=merged_docs,
    parents=["BM25", "Vector"],
)

# Cross-encoder reranking after merge
ce_step = PipelineStep(
    name="Cross-Encoder",
    ranks=rng.permutation(len(merged_docs)),
    chunk_labels=merged_docs,
    parents=["RRF Merge"],
)

merge_rf = MergeRankFlow([bm25_step, vector_step, rrf_step, ce_step])

### Visualize the pipeline DAG

In [ ]:
merge_rf.plot(top_k=10, relevant_chunks=["doc_0", "doc_3", "doc_7"])

### Overlap analysis

At merge points, how much do the branches overlap? How many documents are exclusive to each branch?

In [ ]:
for oa in merge_rf.overlap_analysis(k=10):
    print(f"Merge: {oa['merge_step']}")
    print(f"  Parents: {oa['parents']}")
    print(f"  Shared (top-10): {oa['shared_count']}")
    for parent, count in oa['exclusive_counts'].items():
        print(f"  Exclusive to {parent}: {count}")
    print(f"  Total unique: {oa['total_unique']}")

### Rank correlation between branches

High Spearman correlation between branches means they retrieve similarly -- potentially redundant. Low correlation means they are complementary.

In [ ]:
for rc in merge_rf.rank_correlation():
    rho = rc['spearman_rho']
    rho_str = f"{rho:.3f}" if rho is not None else "N/A (too few common docs)"
    print(f"{rc['parent_pair'][0]} vs {rc['parent_pair'][1]}: "
          f"rho={rho_str} ({rc['common_docs']} common docs)")

---

**End of tutorial series.** For a comprehensive single-file demo, see [demo.ipynb](../demo.ipynb).